# Synthetic Dataset Generation for Mountain NER

## Overview
This notebook handles the creation of a custom synthetic dataset for training a Named Entity Recognition (NER) model. 
The objective is to train a model to accurately identify mountain names in unstructured text. 

 We achieve this by:
1. Loading external vocabularies and sentence templates.
2. Utilizing `spaCy`'s `EntityRuler` for automatic rule-based annotation.
3. Generating a balanced dataset of positive and negative examples.
4. Converting the text into the standard BIO (Begin, Inside, Outside) tagging format required for training transformer models.

In [1]:
import os
import spacy
import random
import pandas as pd
import json

# Fix seed for reproducibility (important for testing and validation)
random.seed(42)

def load_list_from_txt(filepath):
    """
    Helper function to read items from a .txt file.
    Assumes one item per line. Ignores empty lines.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Error: The file '{filepath}' was not found.")
        
    with open(filepath, 'r', encoding='utf-8') as f:
        # Strip removes leading/trailing whitespace and newline characters
        return [line.strip() for line in f.readlines() if line.strip()]

## 1. Load External Resources and Initialize Pipeline
We dynamically load our mountain vocabulary and sentence templates from `.txt` files. 
Then, we configure a blank `spaCy` English pipeline and attach an `EntityRuler` to automatically locate and label the mountains in our generated texts.

In [ ]:
# Initialize a blank spaCy model for English tokenization
nlp = spacy.blank("en")

# Load vocabulary and templates dynamically from external .txt files
mountain_names = load_list_from_txt("data/mountain_names.txt")
positive_templates = load_list_from_txt("data/templates.txt")
negative_templates = load_list_from_txt("data/negative_templates.txt")

# Combine templates
all_templates = positive_templates + negative_templates

# Add EntityRuler to the spaCy pipeline
# This automatically finds mountains in text and assigns the "MOUNTAIN" label
ruler = nlp.add_pipe("entity_ruler")
patterns = [{"label": "MOUNTAIN", "pattern": mountain} for mountain in mountain_names]
ruler.add_patterns(patterns)

## 2. Raw Sentence Generation
We generate 6,500 sentences by randomly combining our mountain names with both positive (mountain-related) and negative (unrelated) templates. The Python `format` method handles the string interpolation dynamically.

In [3]:
# Generate raw sentences
TOTAL_SAMPLES = 6500
generated_sentences = []

for _ in range(TOTAL_SAMPLES):
    mountain_choice = random.choice(mountain_names)
    template_choice = random.choice(all_templates)
    
    # Python's .format() safely ignores {mountain} if a negative template is selected
    try:
        sentence = template_choice.format(mountain=mountain_choice)
    except KeyError:
        sentence = template_choice
        
    generated_sentences.append(sentence)

## 3. Tokenization and BIO Tag Extraction
Transformer models require token-level annotations. We pass our generated sentences through the `spaCy` pipeline to tokenize the text and extract the corresponding BIO (Begin-Inside-Outside) tags for our `MOUNTAIN` entities.

In [4]:
# Tokenization and extraction of BIO tags
dataset_records = []

for text in generated_sentences:
   doc = nlp(text)
   
   tokens = []
   ner_tags = []
   
   for token in doc:
      tokens.append(token.text)
      
      # spaCy automatically formats IOB tags (B, I, O)
      if token.ent_iob_ != "O":
         # Format tag as B-MOUNTAIN or I-MOUNTAIN
         ner_tags.append(f"{token.ent_iob_}-{token.ent_type_}")
      else:
         ner_tags.append("O")
         
   dataset_records.append({
      "tokens": tokens,
      "ner_tags": ner_tags
   })

## 4. Validation and Export
Finally, we review a few samples to ensure the tokenization and tagging align correctly, and save the finalized dataset to a JSON file for the fine-tuning stage.

In [5]:
# Review the results
df = pd.DataFrame(dataset_records)

print("Sample of generated data:")
for i in range(3):
    print(f"Tokens: {df.iloc[i]['tokens']}")
    print(f"Tags:   {df.iloc[i]['ner_tags']}\n")

# Save to JSON format
output_filename = "mountain_ner_dataset.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(dataset_records, f, indent=4)
    
print(f"Dataset successfully saved to '{output_filename}'")

Sample of generated data:
Tokens: ['"', 'We', 'spent', 'the', 'night', 'in', 'a', 'small', 'cabin', 'overlooking', '#', 'Antarctica', '.', '"', ',']
Tags:   ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-MOUNTAIN', 'I-MOUNTAIN', 'O', 'O', 'O']

Tokens: ['"', 'We', 'watched', 'the', 'stars', 'above', '"', 'Cho', 'Oyu', '"', ',', '"', 'Dhaulagiri', '"', ',', '"', 'Manaslu', '"', ',', '"', 'Nanga', 'Parbat', '"', ',', 'all', 'night', '.', '"', ',']
Tags:   ['O', 'O', 'O', 'O', 'O', 'O', 'B-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'I-MOUNTAIN', 'O', 'O', 'O', 'O', 'O']

Tokens: ['"', 'Every', 'autumn', ',', 'thousands', 'of', 'visitors', 'come', 'to', 'admire', '"', 'Mount', 'Whitney', '"', ',', '"', 'Mount', 'Shasta', '"', ',', '"', 'Longs', 'Peak', '"', ',', '.', '"', ',']
Tags:   ['O', 'O'